In [53]:
import pandas as pd
from langchain_dartmouth.llms import ChatDartmouthCloud
from langchain_core.prompts import ChatPromptTemplate

from joblib import Parallel, delayed
from tqdm.auto import tqdm

In [54]:
industry_gender = pd.read_csv("../data/derived/industry_gender_share.csv")
industry_gender = industry_gender[
    industry_gender.industry_name != "Total, 16 years and older"
]
industry_gender.head()

,industry_name,hierarchy_level,employment,percentage_female,parent_level_0,parent_level_1,parent_level_2,parent_level_3
1,"Agriculture, forestry, fishing, and hunting",0,2349,27.7,NaN,NaN,NaN,NaN
2,Crop production,1,1186,28.9,"Agriculture, forestry, fishing, and hunting",NaN,NaN,NaN
3,Animal production and aquaculture,1,832,28.1,"Agriculture, forestry, fishing, and hunting",NaN,NaN,NaN
4,Support activities for agriculture and forestry,1,145,33.9,"Agriculture, forestry, fishing, and hunting",NaN,NaN,NaN
5,"Forestry, except logging",1,53,25.5,"Agriculture, forestry, fishing, and hunting",NaN,NaN,NaN


In [55]:
# Build industry label with hierarchy
industry_gender["industry_full"] = industry_gender[
    [
        "parent_level_0",
        "parent_level_1",
        "parent_level_2",
        "parent_level_3",
        "industry_name",
    ]
].apply(lambda row: " - ".join(row.dropna().values), axis="columns")

industry_gender.head()

,industry_name,hierarchy_level,employment,percentage_female,parent_level_0,parent_level_1,parent_level_2,parent_level_3,industry_full
1,"Agriculture, forestry, fishing, and hunting",0,2349,27.7,NaN,NaN,NaN,NaN,"Agriculture, forestry, fishing, and hunting"
2,Crop production,1,1186,28.9,"Agriculture, forestry, fishing, and hunting",NaN,NaN,NaN,"Agriculture, forestry, fishing, and hunting - ..."
3,Animal production and aquaculture,1,832,28.1,"Agriculture, forestry, fishing, and hunting",NaN,NaN,NaN,"Agriculture, forestry, fishing, and hunting - ..."
4,Support activities for agriculture and forestry,1,145,33.9,"Agriculture, forestry, fishing, and hunting",NaN,NaN,NaN,"Agriculture, forestry, fishing, and hunting - ..."
5,"Forestry, except logging",1,53,25.5,"Agriculture, forestry, fishing, and hunting",NaN,NaN,NaN,"Agriculture, forestry, fishing, and hunting - ..."


In [56]:
us_census_industries = industry_gender.industry_full.values.tolist()

In [57]:
company_profiles = pd.read_csv("../data/derived/company_profiles.csv")

In [58]:
company_profiles.head()

,id,company_name,scraped_linkedin_url,name,country_code,locations,followers,employees_in_linkedin,about,specialties,...,stock_info,get_directions_url,description,additional_info,additional_information,country_codes_array,alumni,alumni_information,website_simplified,unformatted_about
0,verizon-labs,Verizon Labs,https://www.linkedin.com/company/verizon-labs,Verizon Labs,NaN,[],5157,84,NaN,NaN,...,NaN,"[{""directions_url"":""https://www.bing.com/maps?...","Verizon Labs | 5,157 followers on LinkedIn.",NaN,Additional jobs info: Network Operations Cente...,NaN,NaN,NaN,verizon.com,NaN
1,renew,ReNew Power,https://in.linkedin.com/company/renew,ReNew,IN,"[""Commercial Block, Zone 6, Golf Course Road D...",376890,4555,ReNew is a leading decarbonisation solutions p...,"Renewable Energy, Solar Energy , Wind Energy ,...",...,NaN,"[{""directions_url"":""https://www.bing.com/maps?...","ReNew | 376,890 followers on LinkedIn. ReNew i...",NaN,"Additional jobs info: Graduate Engineer (29,65...","[""IN""]",NaN,NaN,renew.com,\n ReNew is a leading decarbonisa...
2,citi,Citibank,https://www.linkedin.com/company/citi,Citi,"US,CA,MY,PA,PE,MX,PH,HK,AE,GB,CO,IN,IL,TW","[""388 Greenwich Street New York, New York 1001...",4604840,193802,Citi's mission is to serve as a trusted partne...,"Banking, Commercial Banking, Investment Bankin...",...,"{""datetime"":""April 21, 2025"",""id"":""C"",""stock_e...","[{""directions_url"":""https://www.bing.com/maps?...","Citi | 4,604,840 followers on LinkedIn. Citi&#...",NaN,"Additional jobs info: Citi (1,410 open jobs). ...","[""US"",""CA"",""MY"",""PA"",""PE"",""MX"",""PH"",""HK"",""AE"",...",NaN,NaN,citigroup.com,\n Citi's mission is to serve as ...
3,epic1979,Epic Systems Corp,https://www.linkedin.com/company/epic1979,Epic,US,"[""1979 Milky Way Verona, WI 53593, US""]",831666,15723,Join us in our mission to help the world get w...,"healthcare, emr, ehr, phr, and software",...,NaN,"[{""directions_url"":""https://www.bing.com/maps?...","Epic | 831,666 followers on LinkedIn. ...with ...",NaN,"Additional jobs info: Epic (49,985 open jobs)....","[""US""]",NaN,NaN,epic.com,\n Join us in our mission to help...
4,dc-energy-llc,DC Energy,https://www.linkedin.com/company/dc-energy-llc,DC Energy,US,"[""1600 Tysons Blvd Fifth Floor McLean, Virgini...",4109,105,DC Energy is a proprietary trading firm that f...,NaN,...,NaN,"[{""directions_url"":""https://www.bing.com/maps?...","DC Energy | 4,109 followers on LinkedIn. DC En...",NaN,"Additional jobs info: Analyst (36,938 open job...","[""US""]",NaN,NaN,dc-energy.com,\n DC Energy is a proprietary tra...


In [59]:
company_profiles = company_profiles[
    [
        "id",
        "company_name",
        "about",
        "specialties",
        "organization_type",
        "industries",
        "unformatted_about",
    ]
]

In [60]:
from langchain_core.output_parsers import JsonOutputParser

In [61]:
def get_census_industry(company_profile: pd.Series) -> dict:
    llm = ChatDartmouthCloud(
        model_name="openai.gpt-4.1-mini-2025-04-14",
        max_tokens=1024,
    )
    output_schema = """```json
{
  "type": "object",
  "properties": {
      "id": {
      "type": "string",
      "description": "Unique identifier for the organization"
    },
      "name": {
      "type": "string",
      "description": "Name of the organization"
    },
      "assessment": {
      "type": "string",
      "description": "Detailed assessment or description of the organization"
    },
      "census_industry": {
      "type": "string",
      "description": "The census industry classification"
    }
  },
  "required": ["id", "name", "assessment", "census_industry"],
  "additionalProperties": false
}
```
"""

    industry_prompt = ChatPromptTemplate(
        [
            (
                "system",
                "Your task is to identify the best matching US Census"
                "industry category for a given company. Discuss the data "
                "given in the company profile before responding with your "
                "final decision with a valid JSON object using the following schema:"
                f"\n{output_schema}\n"
                "Your assessment should be as specific to the company as possible. "
                "For example, if the company is a subsidiary of some other company, "
                "match the industry based on the subsidiary's industry, not the parent company's. "
                "The available US Census industries are:\n\n{{us_census_industries}}.",
            ),
            ("human", "Here is the company profile: \n\n {{company_profile}}"),
        ],
        template_format="jinja2",
    )
    industry_mapper = industry_prompt | llm | JsonOutputParser()

    return industry_mapper.invoke(
        input={
            "us_census_industries": us_census_industries,
            "company_profile": company_profile.to_json(),
        }
    )

In [62]:
results = []

In [68]:
import json
from pathlib import Path
from joblib import Parallel, delayed


def process_single_company(idx, company_profile, results_dir):
    """Process one company and save result immediately"""
    result_file = Path(results_dir) / f"result_{idx}.json"

    # Skip if already processed
    if result_file.exists():
        print(f"Skipping {idx} (already processed)")
        return {"idx": idx, "status": "skipped"}

    try:
        response = get_census_industry(company_profile)

        # Save immediately
        with open(result_file, "w") as f:
            json.dump({"idx": idx, "result": response}, f, indent=2)

        return {"idx": idx, "status": "success", "result": response}

    except Exception as e:
        # Save error info
        error_file = Path(results_dir) / f"error_{idx}.json"
        with open(error_file, "w") as f:
            json.dump({"idx": idx, "error": str(e)}, f, indent=2)

        return {"idx": idx, "status": "failed", "error": str(e)}


def process_parallel_with_saves(company_profiles, results_dir="results"):
    """Process in parallel with individual saves"""
    Path(results_dir).mkdir(exist_ok=True)

    company_data = [(idx, row) for idx, row in company_profiles.iterrows()]

    # Process in parallel, each saving its own result
    results = Parallel(n_jobs=-1)(
        delayed(process_single_company)(idx, company_profile, results_dir)
        for idx, company_profile in tqdm(company_data, desc="Processing companies")
    )

    # Summarize results
    successful = [r for r in results if r["status"] == "success"]
    failed = [r for r in results if r["status"] == "failed"]
    skipped = [r for r in results if r["status"] == "skipped"]

    print(f"✅ Successful: {len(successful)}")
    print(f"❌ Failed: {len(failed)}")
    print(f"⏭️ Skipped: {len(skipped)}")

    return results


results = process_parallel_with_saves(company_profiles)

Skipping 1 (already processed)
Skipping 5 (already processed)
Skipping 2 (already processed)
Skipping 4 (already processed)
Skipping 8 (already processed)
Skipping 0 (already processed)
Skipping 6 (already processed)
Skipping 9 (already processed)
Skipping 10 (already processed)
Skipping 11 (already processed)
Skipping 12 (already processed)
Skipping 13 (already processed)
Skipping 7 (already processed)
Skipping 14 (already processed)
Skipping 15 (already processed)
Skipping 16 (already processed)
Skipping 17 (already processed)
Skipping 18 (already processed)
Skipping 19 (already processed)
Skipping 20 (already processed)
Skipping 21 (already processed)
Skipping 22 (already processed)
Skipping 23 (already processed)
Skipping 24 (already processed)
Skipping 25 (already processed)
Skipping 26 (already processed)
Skipping 27 (already processed)
Skipping 28 (already processed)
Skipping 29 (already processed)
Skipping 30 (already processed)
Skipping 3 (already processed)
Skipping 31 (alrea























































































































































































































































































































































































































































































































































































































































Processing companies: 100%|██████████| 2428/2428 [22:55<00:00,  1.77it/s]


✅ Successful: 2272
❌ Failed: 43
⏭️ Skipped: 113


In [85]:
results = pd.DataFrame.from_records(
    [json.load(file.open()) for file in Path("results").glob("*.json")]
)
results = pd.json_normalize(results.result)
results

,id,name,assessment,census_industry
0,shefly,SheFly Apparel,The company name 'SheFly Apparel' suggests inv...,Wholesale and retail trade - Retail trade - Cl...
1,placepass,PlacePass,PlacePass is a travel technology company that ...,Professional and business services - Managemen...
2,harmonizehq,Harmonize,Harmonize is a privately held company speciali...,Information - Software publishers
3,sanford-c.-bernstein-limited,Sanford C. Bernstein,Sanford C. Bernstein operates in the financial...,Financial activities - Finance and insurance -...
4,carnegieendowment,Carnegie Endowment for International Peace,Carnegie Endowment for International Peace is ...,Professional and business services - Professio...
...,...,...,...,...
2423,zearn,Zearn Inc.,Zearn Inc. is a nonprofit organization providi...,Education and health services - Educational se...
2424,alaska-airmen-s-association,Alaska Airmen Association,The Alaska Airmen Association appears to be a ...,"Other services - Other services, except privat..."
2425,convoy-inc,Convoy,"Convoy operates in the transportation, logisti...",Transportation and utilities - Transportation ...
2426,the-urban-grape,The Urban Grape,The Urban Grape is a specialty retail store se...,Wholesale and retail trade - Retail trade - Be...


In [86]:
results = results.merge(
    right=industry_gender[["industry_full", "percentage_female"]],
    how="left",
    left_on="census_industry",
    right_on="industry_full",
).drop(columns="industry_full")

In [99]:
results = (
    results.merge(right=company_profiles, on="id", how="left")
    .drop(columns="company_name")
    .drop_duplicates()
)

In [103]:
results.to_csv("../data/derived/company_mapping.csv", index=False)